# fMRI Denoising Pipeline - Example Usage

This notebook demonstrates how to use the denoising pipeline to extract time-series from BOLD fMRI data.

In [1]:
import sys
sys.path.append('..')  # Add parent directory to path

from denoising import DenoisingPipeline
from denoising.config import load_config
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Load Configuration

First, load the configuration file that defines atlas, denoising, and confounds parameters.

In [2]:
config = load_config('../configs/strategy_4.yaml')



print(f"Atlas: {config.atlas.name} with {config.atlas.n_regions} regions")
print(f"Smoothing FWHM: {config.denoising.smoothing_fwhm} mm")
print(f"Standardization: {config.denoising.standardize}")

ValueError: Configuration validation failed: 2 validation errors for PipelineConfig
denoising.t_r.int
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='none', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
denoising.t_r.float
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='none', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing

In [17]:
l = ['a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03',
       'a_comp_cor_04', 'a_comp_cor_05', 'a_comp_cor_06', 'a_comp_cor_07',
       'a_comp_cor_08', 'a_comp_cor_09']

for i in l:
    print(f'    - "{i}"')

    - "a_comp_cor_00"
    - "a_comp_cor_01"
    - "a_comp_cor_02"
    - "a_comp_cor_03"
    - "a_comp_cor_04"
    - "a_comp_cor_05"
    - "a_comp_cor_06"
    - "a_comp_cor_07"
    - "a_comp_cor_08"
    - "a_comp_cor_09"


## 2. Initialize Pipeline

Create a `DenoisingPipeline` instance with the configuration.

In [7]:
pipeline = DenoisingPipeline(config)
print("Pipeline initialized successfully")

Pipeline initialized successfully


## 3. Process a Single Subject

Specify paths to BOLD and confounds files, then run the pipeline.

In [4]:
# Update these paths to your data
bold_path = r"C:\Users\ТМ\Desktop\test_func\func\sub-29864_ses-1_task-rest_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
confounds_path = r"C:\Users\ТМ\Desktop\test_func\func\sub-29864_ses-1_task-rest_run-1_desc-confounds_timeseries.tsv"
output_dir = "../output"

# Run the pipeline
output_file = pipeline.process_subject(bold_path, confounds_path, output_dir)
print(f"Output saved to: {output_file}")

Brainnetome atlas: 3.99kB [00:00, ?B/s]
Brainnetome labels: 3.99kB [00:00, 2.69MB/s]


ImageFileError: File C:/Users/ТМ/nilearn_data/brainnetome/BN_Atlas_246_2mm.nii.gz is not a gzip file

## 4. Load and Inspect Results

Load the extracted time-series and explore the data.

In [ ]:
# Load the time-series
timeseries = pd.read_csv(output_file)
print(f"Time-series shape: {timeseries.shape}")
print(f"\nFirst 5 rows:")
timeseries.head()

In [ ]:
# Display region names
print("Brain regions:")
for i, region in enumerate(timeseries.columns[:10]):  # Show first 10
    print(f"  {i+1}. {region}")
print(f"  ... and {len(timeseries.columns) - 10} more")

## 5. Visualize Time-Series

Plot time-series for a few regions.

In [ ]:
# Plot first 5 regions
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

for i, ax in enumerate(axes):
    region = timeseries.columns[i]
    ax.plot(timeseries[region].values)
    ax.set_ylabel('Signal')
    ax.set_title(region)

axes[-1].set_xlabel('Timepoint')
plt.tight_layout()
plt.show()

## 6. Correlation Matrix

Compute and visualize the correlation matrix between regions.

In [ ]:
# Compute correlation matrix (subset of regions for visualization)
n_regions_plot = 50
corr_matrix = timeseries.iloc[:, :n_regions_plot].corr()

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title(f'Correlation Matrix (First {n_regions_plot} Regions)')
plt.tight_layout()
plt.show()

## 7. Batch Processing

Process multiple subjects using a list of file paths.

In [ ]:
# Define subjects to process
subjects = [
    {
        "bold_path": "path/to/subject1_bold.nii.gz",
        "confounds_path": "path/to/subject1_confounds.tsv"
    },
    {
        "bold_path": "path/to/subject2_bold.nii.gz",
        "confounds_path": "path/to/subject2_confounds.tsv"
    }
]

# Run batch processing
# outputs = pipeline.process_batch(subjects, output_dir)
# print(f"Processed {len([o for o in outputs if o])} subjects successfully")

## 8. Custom Configuration

You can modify the configuration for different analysis needs.

In [ ]:
# Example: Create a custom config with different atlas
from denoising.config.schemas import PipelineConfig

custom_config = PipelineConfig(
    atlas={
        "name": "schaefer_2018",
        "resolution": 1,
        "n_regions": 200
    },
    denoising={
        "smoothing_fwhm": 8.0,
        "detrend": True,
        "standardize": "zscore"
    },
    confounds={
        "strategy": "simple",
        "columns": []
    }
)

print(f"Custom config: {custom_config.atlas.n_regions} regions at {custom_config.atlas.resolution}mm")